# 05 — Time-series readiness dan forecasting baseline

## Pertanyaan

Apakah seri JISDOR bulanan siap untuk pemodelan deret waktu, dan seberapa besar error baseline naïve pada periode holdout historis?

## Metode

Kesiapan diperiksa dari periode duplikat dan bulan yang hilang. Maksimal 24 bulan terakhir dipakai sebagai holdout berbasis waktu. Baseline lag-1 memakai nilai aktual bulan sebelumnya dan seasonal-naïve lag-12 memakai bulan yang sama tahun sebelumnya. MAE, RMSE, dan MAPE dihitung; observasi aktual nol dikeluarkan hanya dari penyebut MAPE. Ini evaluasi historis rolling one-step, bukan proyeksi masa depan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analytics.descriptive.data_access import DATASET_SOURCES
from analytics.descriptive.notebook_support import insight, prepare_notebook, save_figure
from analytics.forecasting.baseline import naive_holdout_backtest, time_series_readiness

data = prepare_notebook()
series = data.monetary[["observation_date", "jisdor_idr_per_usd"]].copy()
series["jisdor_idr_per_usd"] = pd.to_numeric(series["jisdor_idr_per_usd"], errors="raise")
readiness = time_series_readiness(
    series, date_column="observation_date", value_column="jisdor_idr_per_usd"
)
display(pd.DataFrame([{key: value for key, value in readiness.items() if key != "missing_periods"}]))
if readiness["missing_periods"]:
    display(pd.DataFrame({"missing_period": readiness["missing_periods"]}))
    raise ValueError("Backtest dihentikan karena seri JISDOR memiliki bulan yang hilang")

## Hasil

In [ ]:
test_periods = min(24, max(3, len(series) // 5))
backtests = {
    "naive_lag_1": naive_holdout_backtest(
        series,
        date_column="observation_date",
        value_column="jisdor_idr_per_usd",
        test_periods=test_periods,
        prediction_lag=1,
    )
}
if len(series) > test_periods + 12:
    backtests["seasonal_naive_lag_12"] = naive_holdout_backtest(
        series,
        date_column="observation_date",
        value_column="jisdor_idr_per_usd",
        test_periods=test_periods,
        prediction_lag=12,
    )
metrics = pd.DataFrame(
    [{"model": name, **result.metrics} for name, result in backtests.items()]
).sort_values("mae", ignore_index=True)
display(metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
history_start = backtests["naive_lag_1"].test_start - pd.DateOffset(months=24)
history = series.loc[series["observation_date"].ge(history_start)]
ax.plot(history["observation_date"], history["jisdor_idr_per_usd"], color="black", label="Aktual")
for model_name, result in backtests.items():
    ax.plot(result.predictions["observation_date"], result.predictions["prediction"], linestyle="--", label=model_name)
ax.axvline(backtests["naive_lag_1"].test_start, color="grey", linestyle=":", label="Awal holdout")
ax.set(title="Backtest historis baseline JISDOR", xlabel="Bulan", ylabel="IDR per USD")
ax.legend()
fig.tight_layout()
save_figure(fig, "05_jisdor_baseline_backtest.png")
display(fig)
plt.close(fig)

In [ ]:
best = metrics.iloc[0]
holdout = backtests[best["model"]].predictions
display(insight(
    f"Pada holdout {test_periods} bulan, baseline dengan MAE terendah adalah {best['model']} (MAE {best['mae']:.2f}, RMSE {best['rmse']:.2f}, MAPE {best['mape_percent']:.2f}%). Seri memiliki {readiness['missing_period_count']} bulan hilang di antara awal dan akhir periode.",
    frame=series,
    source=DATASET_SOURCES["monetary"],
    limitation="Evaluasi memakai holdout historis dan prediksi rolling yang dapat menggunakan aktual periode holdout sebelumnya. Hasil ini belum memiliki confidence interval, belum dibandingkan dengan model statistik, dan bukan forecast masa depan.",
))
display(holdout.tail())

## Interpretasi

Baseline menetapkan tingkat kesulitan minimum yang harus dikalahkan model Fase 6 pada data out-of-sample. Model yang lebih rumit belum layak digunakan jika error-nya tidak lebih baik secara material.

## Keterbatasan

Notebook ini belum memilih model produksi, membuat confidence interval, menetapkan threshold publikasi, atau menghasilkan nilai masa depan. Langkah tersebut termasuk advanced analytics Fase 6 dan harus tetap dibandingkan dengan baseline ini.